In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import sys
from scipy.signal import find_peaks, peak_prominences, peak_widths
from scipy import ndimage
from glob import glob
from PIL import Image

def imshow_cv2(img):
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

def str2list(s):
    return list(map(float, s.split()))

def list2matrix(l):
    return np.array(l)


PARAMETERS_CONFIG = {
    # ------------------- Preprocessing Parameters ------------------ #
    "clahe_clip_limit": 2.0, # Contrast limit for CLAHE, higher values give more contrast.
    "clahe_tile_grid_size": (8, 8), # Size of the grid for CLAHE. Larger values may lead to better contrast but can also introduce noise.
    "blur_sigma": 5.0, # Standard deviation for Gaussian blur
    
    # ------------------- Peak Detection Parameters ------------------ #
    "min_peak_distance": 5, # Minimum distance between peaks in the contour distance profile. Adjust based on expected tooth spacing.
    "peak_local_window": 9, # Size of the local window for peak filtering. 
    "min_peak_width": 2.0,
    "max_relative_radius_jump": 0.40,
    "min_peak_prominence": 1.5 # Minimum prominence of peaks to be considered
}
    

**1. Image Preprocessing**

Preprocessing of the image, which includes contranst enhancement and noise reduction with the aim of improving the segmentation performance.

In [ ]:
def preprocess_image(img, parameters_config=PARAMETERS_CONFIG):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 1. Watermark Removal (210 - 255 gray range to mask)
    mask = cv2.inRange(gray, 210, 255)
    gray[mask > 0] = 255
   
   # 2. CLAHE Histogram Equalization
    clahe = cv2.createCLAHE(clipLimit=parameters_config["clahe_clip_limit"], tileGridSize=parameters_config["clahe_tile_grid_size"])
    gray = clahe.apply(gray)
   
    # Min Filter
    blurred = ndimage.minimum_filter(gray, size=3)
    
    
    return blurred

# Test the preprocessing function
image_path = './data/train/train_3.jpg'
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


preprocessed_img = preprocess_image(img, parameters_config=PARAMETERS_CONFIG)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title('Original Image')
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title('Preprocessed Image')
plt.imshow(preprocessed_img, cmap='gray')
plt.axis('off')
plt.show()

**2. Contour Detection**

Segmentation of the gear from the background and detection of its contour.

In [ ]:
def segment_gear(image):
    # Binarize the image using OTSU Thresholding
    _, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # invert the image if the background is not black (i.e. if the pixel at (1,1) is white)
    if binary[1, 1] == 255:
        binary = cv2.bitwise_not(binary)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    
    # Apply morphological gradient to enhance the edges of the gear
    binary = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel, iterations=1)
    
    # close small holes in the binary image
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)
    
    return binary



def extract_contours(image):
    
    binary = segment_gear(image)
    
    # Find contours using RETR_EXTERNAL mode
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    gear_contour = max(contours, key=cv2.contourArea)

    
    return gear_contour, binary

# Test the contour extraction function
gear_contour, binary = extract_contours(preprocessed_img)

cv2.drawContours(img, [gear_contour], -1, (0, 255, 0), 2)
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title('Binary Image')
plt.imshow(binary, cmap='gray')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title('Contours')
plt.imshow(img)
plt.axis('off')
plt.show()



**3. Gear Detection**

Detection of the gear teeth by analyzing peak locations in the contour.

In [ ]:
def find_best_peak_sequence(peaks_data):
    """
    HELPER FUNCTION:
    Receives a list of tuples (PeakIndex, Prominence).
    Returns the start and end indices of the best (clean and strong) sequence of peaks.
    """
    # 1. Extract prominences only to compute the threshold
    prominences = np.array([p[1] for p in peaks_data])
    
    # 2. Define a dynamic threshold (e.g., 75% of the median)
    # This value separates "true" teeth from background noise
    soglia = np.median(prominences) * 0.75
    
    # 3. Identify peaks above threshold (boolean mask)
    validi = prominences >= soglia
    
    # 4. Algorithm to find the longest contiguous True subsequence
    max_len = 0
    current_len = 0
    best_range = (0, 0)
    temp_start = 0
    
    for i, is_valid in enumerate(validi):
        if is_valid:
            if current_len == 0:
                temp_start = i
            current_len += 1
            if current_len > max_len:
                max_len = current_len
                best_range = (temp_start, i)
        else:
            current_len = 0
            
    return best_range

def find_contour_peaks(dists, parameters_config=PARAMETERS_CONFIG):
    
    peaks, _ = find_peaks(dists, distance=parameters_config["min_peak_distance"], prominence=parameters_config["min_peak_prominence"])
    
    if len(peaks) == 0:
        print("No peaks found with the given parameters.")
        return [], dists

    # Compute the prominence of each peak, which measures how much the peak stands out from the surrounding baseline
    # Higher prominence means the peak is more distinct from its neighbors (better candidate for a tooth tip)
    prominences = peak_prominences(dists, peaks)[0]
    
    # find the best sequence of peaks based on their prominences
    peaks_data = list(zip(peaks, prominences))
    best_start, best_end = find_best_peak_sequence(peaks_data)
    best_indexes = peaks[best_start:best_end + 1]
    
    # Find the mean distance between the best peaks to filter out any remaining outliers
    beast_mean_distance = int(np.mean(np.diff(best_indexes)))

    first_peak = peaks[0]
    last_peak = peaks[-1]
    # add the first peak if not already present
    if first_peak in range(1, 5) or last_peak in range(len(gear_contour) - 5, len(gear_contour) - 1):
        pass # the first peak is already included in the first or last points
    else:
        peaks = np.insert(peaks, 0, 0)  # add 0 at the beginning of the peaks
        prominences = np.insert(prominences, 0, prominences[0])  # add the prominence of the first peak at the beginning of the prominences
    
    # filter out the peaks if it's distance from the previous one is too different from the mean distance of the best peaks
    filtered_peaks = []
    for i in range(len(peaks)):
        if i == 0:
            filtered_peaks.append(peaks[i])
        else:
            distance_from_previous = peaks[i] - peaks[i-1]
            if distance_from_previous >= beast_mean_distance - 20 and distance_from_previous <= beast_mean_distance + 20:
                filtered_peaks.append(peaks[i])
    
    filtered_prominences = [prominences[i] for i in range(len(peaks)) if peaks[i] in filtered_peaks]

    # add missing peaks based on the mean distance of the best peaks
    # for each peak if the next peak is too far from the current one, add a new peak at the expected position based on the mean distance
    final_peaks = []
    final_prominences = []
    for i in range(len(filtered_peaks) - 1):
        final_peaks.append(filtered_peaks[i])
        final_prominences.append(filtered_prominences[i])
        distance_to_next = filtered_peaks[i + 1] - filtered_peaks[i]
        if distance_to_next > beast_mean_distance + 20:
            expected_peak = filtered_peaks[i] + beast_mean_distance + np.random.randint(-10, 10) # add some random noise to the expected position of the new peak
            final_peaks.append(expected_peak)
            final_prominences.append(filtered_prominences[i])
    final_peaks.append(filtered_peaks[-1])  # add the last peak
    final_prominences.append(filtered_prominences[-1])  # add the prominence of the last peak
    
    return final_peaks, final_prominences
    
    
    
def map_peaks_to_contour(contour, peaks, gear_center, binary=None, base_offset=7, max_offset=15):
    """
    Convert peak indices on the distance profile into stable (x, y) tooth points.
    """
    
    peak_indices = [int(p) for p in peaks]

    # Flatten contour to Nx2 and read geometry basics
    c = contour[:, 0, :].astype(np.float32)  # contour points as (x, y)
    N = len(c)                                # number of contour samples
    cx, cy = float(gear_center[0]), float(gear_center[1])  # gear centroid

    # Optional: distance transform map, used to pick safer inner points
    dt = None
    if binary is not None:
        mask = (binary > 0).astype(np.uint8) * 255
        dt = cv2.distanceTransform(mask, cv2.DIST_L2, 3)

    teeth_points = []
    k = 5  # half-window size for local peak refinement

    for peak_idx in peak_indices:
        # Wrap index to contour length (safe circular indexing)
        i = peak_idx % N

        # 1) Refine the tip around the peak using nearby contour points
        idxs = [(i + t) % N for t in range(-k, k + 1)]
        pts = c[idxs]

        # Weight points by radial distance: farther points influence tip more
        rad = np.linalg.norm(pts - np.array([cx, cy], dtype=np.float32), axis=1)
        w = (rad - rad.min()) + 1e-3
        tip = (pts * w[:, None]).sum(axis=0) / w.sum()
        tip_x, tip_y = float(tip[0]), float(tip[1])

        # 2) Build inward direction (from tip to centroid)
        vx, vy = cx - tip_x, cy - tip_y
        norm = np.hypot(vx, vy)
        if norm < 1e-6:
            continue  # skip degenerate case
        ux, uy = vx / norm, vy / norm

        # 3) Compute adaptive inward push (clamped between min/max offsets)
        local_r = float(np.hypot(tip_x - cx, tip_y - cy))
        offset = int(np.clip(base_offset + 0.04 * local_r, base_offset, max_offset))

        # Default candidate point after inward push
        best_x = int(round(tip_x + ux * offset))
        best_y = int(round(tip_y + uy * offset))

        # 4) If distance transform exists, search along inward ray
        #    and keep the point with maximum clearance from boundaries
        if dt is not None:
            best_score = -1.0
            for t in range(base_offset, max_offset + 1):
                xx = int(round(tip_x + ux * t))
                yy = int(round(tip_y + uy * t))
                if 0 <= xx < dt.shape[1] and 0 <= yy < dt.shape[0]:
                    score = dt[yy, xx]
                    if score > best_score:
                        best_score = score
                        best_x, best_y = xx, yy

        # Final stable point for this tooth
        teeth_points.append((best_x, best_y))

    return teeth_points

In [ ]:
# plot the distances with peaks

M  = cv2.moments(gear_contour)
cx = int(M['m10'] / M['m00'])
cy = int(M['m01'] / M['m00'])

# Compute all distances from centroid to every contour point
dists = np.array([np.linalg.norm(p[0] - [cx, cy]) for p in gear_contour])

peaks, prominences= find_contour_peaks(dists)

print("Peaks:", peaks)
print("prominences:", prominences)

# --- VISUALIZZAZIONE FINALE ---
# Usiamo l'immagine a colori per vedere bene i punti

result_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# Disegniamo il contorno in verde scuro
cv2.drawContours(result_img, [gear_contour], -1, (0, 100, 0), 1)

# Disegniamo il baricentro in blu
cv2.circle(result_img, (cx, cy), 4, (255, 0, 0), -1)

teeth_points = map_peaks_to_contour(gear_contour, peaks, (cx, cy))

# Disegniamo i punti definitivi dei denti in rosso
# aggiungi l'indice del picco vicino al punto disegnato
for i, pt in enumerate(teeth_points):
    cv2.circle(result_img, pt, 6, (255, 0, 0), -1)

plt.figure(figsize=(12, 10))
plt.imshow(result_img)
for i, pt in enumerate(teeth_points):
    plt.annotate(f'{i}', pt, textcoords="offset points", xytext=(0,10), ha='center', color='white', fontsize=12, fontweight='bold', bbox=dict(boxstyle="round,pad=0.1", fc="red", ec="none", alpha=0.8))

plt.figure(figsize=(12, 10))
plt.subplot(2, 1, 1)
plt.title('Distance from Centroid to Contour Points')
plt.plot(dists)
# Aggiungi etichetta della prominence vicino a ogni picco (usando la quota reale del picco)
plt.scatter(peaks, dists[peaks], color='red', s=50, label='Peaks', zorder=3)

for i, peak_idx in enumerate(peaks):
    y = dists[int(peak_idx)]  # altezza reale del picco sulla curva delle distanze
    plt.annotate(
        f'{prominences[i]:.2f}',
        (peak_idx, y),
        textcoords="offset points",
        xytext=(0, 10),
        ha='center',
        color='white',
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.15", fc="red", ec="none", alpha=0.85),
        zorder=4
    )

plt.legend()
plt.subplot(2, 1, 2)
plt.title('Gear Teeths Prediction')
plt.imshow(result_img)
for i, pt in enumerate(teeth_points):
    plt.annotate(f'{i}', pt, textcoords="offset points", xytext=(0,10), ha='center', color='white', fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.1", fc="red", ec="none", alpha=0.8))

plt.show()

In [ ]:
def find_gear_theet(img):
    preprocessed_img = preprocess_image(img)
    gear_contour, binary = extract_contours(preprocessed_img)
    M  = cv2.moments(gear_contour)
    cx = int(M['m10'] / M['m00'])
    cy = int(M['m01'] / M['m00'])
    dists = np.array([np.linalg.norm(p[0] - [cx, cy]) for p in gear_contour])
    final_peaks, dists = find_contour_peaks(dists)
    teeth_points = map_peaks_to_contour(gear_contour, final_peaks, (cx, cy))
    
    return teeth_points

In [ ]:
def score(solution, submission, debug_item=False):
    """
    this metric computes average precision and recall in theet detection
    """
    # TODO: You likely want to delete the row ID column, which Kaggle's system uses to align
    # the solution and submission before passing these dataframes to score().
    del solution['id']
    del submission['id']

    if submission.isnull().values.any():
        raise ParticipantVisibleError("Missing data in submission file")

    average_recall = 0
    average_precision = 0
    for i in range(len(solution )):
                bbox = solution['bbox'][i]
                
                theet=submission['bbox'][i]
                
                bbox=str2list(bbox)
                bbox=np.array(bbox).reshape(-1,4)
                
                theet=str2list(theet)
                theet = np.array(theet).reshape(-1,2).T
            
                true_positive = 0
               

                for j in range(len(bbox)):
                    x_min, y_min, x_max, y_max = bbox[j]
                    dx = (theet[0,:] - x_min) > 0
                    dx &= (x_max - theet[0,:]) > 0
                    dy = (theet[1,:] - y_min) > 0
                    dy &= (y_max - theet[1,:]) > 0
                    if(np.where(dx & dy)[0].shape[0]==1):
                        true_positive+=1

                false_positive=theet.shape[1]-true_positive    
                if(debug_item):
                    print('# of teeth=', len(bbox),', TP=',true_positive,', FP=',false_positive,': precision=',true_positive/theet.shape[1],' recall=',true_positive/len(bbox) )
                average_recall+=(true_positive/len(bbox))
                average_precision+=(true_positive/theet.shape[1])
        
    average_recall=average_recall/len(solution)
    average_precision=average_precision/len(solution)
    return average_recall,average_precision

In [ ]:
# test visualization and scoring metric of training set

train_PATH='./data/train'
df = pd.read_csv(os.path.join(train_PATH,'annotation.csv'))
solution_df = {'id': [], 'bbox': []}
for i in range(len(df)): # loop over training images
    path=os.path.join(train_PATH,df['id'][i])
    img = cv2.imread(path)
    
    #plot gear image and annotated bounding boxes around thetheet
    plt.figure()
    imshow_cv2(img)
    bbox = df['bbox'][i]
    bbox=str2list(bbox)
    bbox=np.array(bbox).reshape(-1,4)
    
    for j in range(len(bbox)):
        x_min, y_min, x_max, y_max = bbox[j]
        plt.plot([x_min,x_max,x_max,x_min,x_min],[y_min,y_min,y_max,y_max,y_min],'r--')

    # DECTOR TO BE DESIGNED
    theet = find_gear_theet(img)

    # reshape the theet list to a 2xN matrix for plotting and csv logging
    theet = list2matrix(theet)

    # plot theer coordinates for debugging
    plt.plot(theet[:,0],theet[:,1],'g.')

    # create solution csv for scoring
    solution_df['id'].append(df['id'][i])
    #flatten teeth coordinates to string for csv logging
    theet = " ".join(map(str, theet.flatten()))
    solution_df['bbox'].append(theet)
    

pd.DataFrame(solution_df).to_csv('train_solution.csv', index=False)

sf = pd.read_csv('train_solution.csv')
average_recall, average_precision = score(df,sf,debug_item=True)
print('Recall=%.3f, Precision=%.3f'%(average_recall,average_precision))



In [ ]:
test_PATH='./data/test'

# read all test images in the test folder
test_images = glob(os.path.join(test_PATH,'*.jpg'))
submission_df = {'id': [], 'bbox': []}
for path in test_images:
    img = cv2.imread(path)
    
    #plot gear image and annotated bounding boxes around thetheet
    plt.figure()
    imshow_cv2(img)

    theet = find_gear_theet(img)

    # reshape the theet list to a 2xN matrix for plotting and csv logging
    theet = list2matrix(theet)

    # plot theer coordinates for debugging
    plt.plot(theet[:,0],theet[:,1],'g.')

    # create solution csv for scoring
    submission_df['id'].append(os.path.basename(path))
    #flatten teeth coordinates to string for csv logging
    theet = " ".join(map(str, theet.flatten()))
    submission_df['bbox'].append(theet)
    
pd.DataFrame(submission_df).to_csv('submission.csv', index=False)